<a href="https://colab.research.google.com/github/niimraabiid/IqbalGPT/blob/main/IqbalGPT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IqbalGPT — Training Notebook

Character-level GPT trained from scratch on Allama Iqbal's *Asrar-i-Khudi*
(R.A. Nicholson's 1920 English translation, public domain via Project Gutenberg),
using Andrej Karpathy's nanoGPT framework.

Full project + README: https://github.com/niimraabiid/IqbalGPT

**Final results:** train loss 0.92, val loss 1.39 (step 2000) — see repo README
for full evaluation and discussion of overfitting onset around step 1500.

### 1. Mount Google Drive

Connects this notebook to your Google Drive so all files/checkpoints persist
permanently, instead of disappearing when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Create a persistent project folder

Makes an `IqbalGPT` folder inside Drive and moves into it — everything
from here on is saved there, not in temporary Colab storage.

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/IqbalGPT', exist_ok=True)
%cd /content/drive/MyDrive/IqbalGPT

### 3. Clone nanoGPT and install dependencies

Downloads Andrej Karpathy's nanoGPT framework and installs the Python
packages it needs (PyTorch, tiktoken, etc.).

In [ ]:
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### 4. Confirm working directory

Sanity check to make sure we're inside `IqbalGPT/nanoGPT` before continuing.

In [ ]:
%pwd

### 5. Confirm the corpus file uploaded correctly

`input.txt` (the cleaned Iqbal text) was uploaded manually via the file
browser into `data/iqbal/` — this just confirms it's there.

In [ ]:
!ls data/iqbal/

### 6. Add the data-preparation script

Copies nanoGPT's character-level tokenization script into our `iqbal`
data folder, next to `input.txt`.

In [ ]:
!mkdir -p data/iqbal
!cp data/shakespeare_char/prepare.py data/iqbal/prepare.py
!ls data/iqbal/

### 7. Tokenize the corpus

Converts the raw text into character-level training data (`train.bin`,
`val.bin`) and records the character vocabulary (`meta.pkl`).

In [ ]:
%cd data/iqbal
!python prepare.py
%cd /content/drive/MyDrive/IqbalGPT/nanoGPT

### 8. Verify the tokenized data

Confirms `train.bin` and `val.bin` were created and checks their size —
163KB train / 18KB validation.

In [ ]:
import os
print("Train size:", os.path.getsize('data/iqbal/train.bin'), "bytes")
print("Val size:", os.path.getsize('data/iqbal/val.bin'), "bytes")

### 9. Configure the model and training run

Defines model size (4 layers, 4 heads, 256-dim embeddings) and training
settings, deliberately scaled down to suit this small (~160KB) corpus
and reduce overfitting risk.

In [ ]:
%%writefile config/train_iqbal.py
out_dir = 'out-iqbal'
eval_interval = 250
eval_iters = 200
log_interval = 10

always_save_checkpoint = True

dataset = 'iqbal'
gradient_accumulation_steps = 1
batch_size = 32
block_size = 128

n_layer = 4
n_head = 4
n_embd = 256
dropout = 0.2

learning_rate = 1e-3
max_iters = 2000
lr_decay_iters = 2000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 200

device = 'cuda'
compile = True

### 10. Train the model

Trains for 2000 steps on a T4 GPU. Checkpoints save to Drive throughout,
so progress survives a disconnect. Final: train loss 0.92, val loss 1.39
(overfitting onset observed around step 1500 — see README for full analysis).

In [ ]:
!python train.py config/train_iqbal.py

### 11. Generate text from the trained model

Loads the trained checkpoint and generates sample Iqbal-style text,
starting from a given prompt word.

In [ ]:
!python sample.py --out_dir=out-iqbal --start="The hour" --num_samples=3 --max_new_tokens=300